# 02.1 – Feature Engineering

Bereinigung und Feature-Aufbereitung: Spalten-Selektion, Duplikat-Filter, FDZ-Missings (< 0) zu Null, Completeness-Score, binäre Flags, Ausreißer-Filter und Target-Clipping.

In [1]:
import polars as pl
import os

FULL_PATH = r"C:\Users\Admin\Desktop\Übertrag\Rental-price-prediction\data\processed\AUTO_WM_Full.parquet"
CLEANED_PATH = r"C:\Users\Admin\Desktop\Übertrag\Rental-price-prediction\data\processed\AUTO_df_cleaned.parquet"

print("Lade Daten...")
df = pl.read_parquet(FULL_PATH)

Lade Daten...


In [2]:
# Spalten-Selektion (immobilientyp entfernt, da Zero-Variance = 4)
keep_cols = [
    "obid", "dupID_gen", "ajahr", "baujahr", "mietekalt", "wohnflaeche", "zimmeranzahl",
    "kid2019", "erg_amd", "ergg_1km", "blid",
    "objektzustand", "heizungsart", 
    "balkon", "einbaukueche", "keller", "aufzug", "garten", "gaestewc",
    "parkplatz", "foerderung",
    "parkplatzpreis", "letzte_modernisierung", "ev_kennwert", "freiab", "amonat",
    "etage", "anzahletagen", "schlafzimmer", "badezimmer",
    "kategorie_Wohnung", "haustier_erlaubt",
    "laufzeittage", "ejahr", "emonat"
]

df_clean = df.select([c for c in keep_cols if c in df.columns])

# 0a. Duplikat-Filter: Nur neueste Inserate behalten (FDZ max dupID_gen pro obid)
if "dupID_gen" in df_clean.columns:
    df_clean = df_clean.sort(["obid", "dupID_gen"]).unique(subset=["obid"], keep="last")
    df_clean = df_clean.drop("dupID_gen")

# 0. FDZ-Missings: Werte < 0 zu Null (verhindert u.a. negative Zimmeranzahlen)
numeric_cols = [c for c in df_clean.columns if df_clean[c].dtype in pl.NUMERIC_DTYPES]

df_clean = df_clean.with_columns([
    pl.when(pl.col(c) >= 0).then(pl.col(c)).otherwise(None).alias(c)
    for c in numeric_cols
])

# 1. Completeness Score (vor der Imputation)
df_clean = df_clean.with_columns([
    (pl.sum_horizontal(pl.all().is_not_null()) / len(df_clean.columns)).alias("completeness_score")
])

# 2. Baujahr-Filter: keine Gebäude vor 1850, max. 2 Jahre in der Zukunft
df_clean = df_clean.filter(
    (pl.col("baujahr") >= 1850) & 
    (pl.col("baujahr") <= pl.col("ajahr") + 2)
)
df_clean = df_clean.with_columns([
    (pl.col("ajahr") - pl.col("baujahr")).alias("alter_bauwerk")
])

# negative Werte auf 0 klemmen (Neubau)
df_clean = df_clean.with_columns([
    pl.col("alter_bauwerk").clip(lower_bound=0)
])

# 3. Binäre Flags (Parkplatz, Modernisierung, Energieausweis)
benno_exprs = []
if "parkplatzpreis" in df_clean.columns and "parkplatz" in df_clean.columns:
    benno_exprs.append((pl.col("parkplatzpreis").is_not_null() | (pl.col("parkplatz") == 1)).fill_null(0).cast(pl.Int8).alias("hat_parkplatz"))
elif "parkplatzpreis" in df_clean.columns:
    benno_exprs.append(pl.col("parkplatzpreis").is_not_null().cast(pl.Int8).alias("hat_parkplatz"))
    
if "letzte_modernisierung" in df_clean.columns:
    benno_exprs.append(pl.col("letzte_modernisierung").is_not_null().cast(pl.Int8).alias("wurde_modernisiert"))
if "ev_kennwert" in df_clean.columns:
    benno_exprs.append(pl.col("ev_kennwert").is_not_null().cast(pl.Int8).alias("hat_energieausweis"))

if benno_exprs:
    df_clean = df_clean.with_columns(benno_exprs)

# 4. freiab in Monate parsen
import re
SOFORT_RE = re.compile(r'sofort|nächstm|naechstm|n.{1,2}chstm|bezugsfrei|sof\.', re.IGNORECASE)
DATE_DMY = re.compile(r'(?<!\d)(\d{1,2})[.\-/](\d{1,2})[.\-/](\d{2,4})(?!\d)')
DATE_YMD = re.compile(r'(?<!\d)(\d{4})[\-/.](\d{1,2})[\-/.](\d{1,2})(?!\d)')
DATE_MY = re.compile(r'(?<!\d)(\d{1,2})[.\-/](\d{4})(?!\d)')
MONTHS_DE = {
    "januar": 1, "jan": 1, "februar": 2, "feb": 2, "märz": 3, "maerz": 3, "mrz": 3, "mär": 3,
    "april": 4, "apr": 4, "mai": 5, "juni": 6, "jun": 6, "juli": 7, "jul": 7,
    "august": 8, "aug": 8, "september": 9, "sep": 9, "sept": 9,
    "oktober": 10, "okt": 10, "november": 11, "nov": 11, "dezember": 12, "dez": 12,
}
MONTH_NAME_RE = re.compile(r'(' + '|'.join(MONTHS_DE.keys()) + r')\w*\.?\s*/?\.?\s*(?:\w*\s*)?(\d{2,4})', re.IGNORECASE)
_ENC_FIX = {'Ã¤': 'ä', 'Ã¶': 'ö', 'Ã¼': 'ü', 'ÃŸ': 'ß'}

def parse_freiab_polars(row):
    val, amonat, ajahr = row["freiab"], row["amonat"], row["ajahr"]
    if val is None or amonat is None or ajahr is None: return None
    s = str(val).strip()
    for bad, good in _ENC_FIX.items(): s = s.replace(bad, good)
    if not s: return None
    if SOFORT_RE.search(s): return 0.0
    year = month = None
    m = DATE_YMD.search(s)
    if m: year, month = int(m.group(1)), int(m.group(2))
    if year is None:
        m = DATE_DMY.search(s)
        if m: month, year = int(m.group(2)), int(m.group(3))
        if year is not None and year < 100: year += 2000
    if year is None:
        m = DATE_MY.search(s)
        if m: month, year = int(m.group(1)), int(m.group(2))
    if year is None:
        m = MONTH_NAME_RE.search(s)
        if m:
            month = MONTHS_DE[m.group(1).lower()]
            year = int(m.group(2))
            if year < 100: year += 2000
    if year is None or month is None: return None
    if not (1 <= month <= 12) or not (1990 <= year <= 2035): return None
    diff = (year - int(ajahr)) * 12 + (month - int(amonat))
    return float(min(max(diff, 0), 24))

if "freiab" in df_clean.columns and "amonat" in df_clean.columns:
    df_clean = df_clean.with_columns([
        pl.struct(["freiab", "amonat", "ajahr"]).map_elements(parse_freiab_polars, return_dtype=pl.Float64).alias("freiab_monate")
    ])

# 5. Outlier Detection & Base Target Calculation
df_clean = df_clean.filter(
    (pl.col("mietekalt") > 50) & (pl.col("mietekalt") < 10000) &
    (pl.col("wohnflaeche") > 10) & (pl.col("wohnflaeche") < 500)
)

df_clean = df_clean.with_columns([
    (pl.col("mietekalt") / pl.col("wohnflaeche")).alias("mietepro_m2")
])

# 6. Target Clipping (Miete pro m2 und Laufzeittage)
if "mietepro_m2" in df_clean.columns:
    q01 = df_clean.select(pl.col("mietepro_m2").quantile(0.01)).item()
    q99 = df_clean.select(pl.col("mietepro_m2").quantile(0.99)).item()
    df_clean = df_clean.with_columns([
        pl.when(pl.col("mietepro_m2") < q01).then(q01)
          .when(pl.col("mietepro_m2") > q99).then(q99)
          .otherwise(pl.col("mietepro_m2")).alias("mietepro_m2")
    ])
    
# Log-Transformation statt hartem Clipping
if "laufzeittage" in df_clean.columns:
    df_clean = df_clean.with_columns([
        pl.col("laufzeittage").cast(pl.Float64).log1p().alias("log_laufzeittage")
    ])

# 7. Imputation: binäre Flags mit 0 füllen, Kategorien/Numerik bleiben null
bin_cols = ["balkon", "einbaukueche", "keller", "aufzug", "garten", "gaestewc", "parkplatz", "foerderung", "haustier_erlaubt"]
df_clean = df_clean.with_columns([
    pl.col(c).fill_null(0).alias(c) for c in bin_cols if c in df_clean.columns
])

df_clean = df_clean.drop([c for c in ["parkplatzpreis", "letzte_modernisierung", "ev_kennwert", "freiab"] if c in df_clean.columns])
if "schlafzimmer" in df_clean.columns and "zimmeranzahl" in df_clean.columns:
    df_clean = df_clean.with_columns([pl.min_horizontal(pl.col("schlafzimmer"), pl.col("zimmeranzahl")).alias("schlafzimmer")])
# 8. Nominale Features als Strings
cat_cols = ['heizungsart', 'objektzustand', 'kategorie_Wohnung', 'blid', 'erg_amd']
df_clean = df_clean.with_columns([pl.col(c).cast(pl.String).alias(c) for c in cat_cols if c in df_clean.columns])

df_clean.write_parquet(CLEANED_PATH)
print(f"{df_clean.height} Zeilen gespeichert.")


C:\Users\Admin\AppData\Local\Temp\ipykernel_27056\2232472891.py:23: DeprecationWarning: `NUMERIC_DTYPES` was deprecated in version 1.0.0. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  numeric_cols = [c for c in df_clean.columns if df_clean[c].dtype in pl.NUMERIC_DTYPES]


12034680 Zeilen gespeichert.
